In [28]:
pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [29]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

i searched for "embedded based labeling sentence transformers" and came across the all-MiniLM-L6-v2 Sentence Transformer, so lets have a look at that


In [30]:
model = SentenceTransformer('all-MiniLM-L6-v2')


In [31]:
dept_labels = [
    "Marketing",
    "Sales",
    "Project Management",
    "IT",
    "Finance",
    "HR",
    "Other"
]

In [32]:
dept_label_embeddings = model.encode(dept_labels)

In [33]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for person_id, cv in enumerate(cvs):
    for job in cv:
        if job["status"] == "ACTIVE":
            jobs.append({
                **job,
                "person_id": person_id
            })

df_active = pd.DataFrame(jobs)
df_active.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0


In [34]:
job_embeddings = model.encode(df_active["position"].astype(str).tolist())

In [35]:
similarity_matrix = cosine_similarity(job_embeddings, dept_label_embeddings)

In [36]:
best_label_idx = similarity_matrix.argmax(axis=1)
df_active["department_pred_embedding"] = [dept_labels[i] for i in best_label_idx]

In [37]:
df_active[["position", "department", "department_pred_embedding"]].head(10)

,position,department,department_pred_embedding
0,Prokurist,Other,HR
1,CFO,Other,Finance
2,Betriebswirtin,Other,IT
3,Prokuristin,Other,HR
4,CFO,Other,Finance
5,Solutions Architect,Information Technology,Project Management
6,Medizintechnik Beratung,Consulting,Marketing
7,Director expansión de negocio.,Business Development,Project Management
8,Gerente comercial,Sales,Marketing
9,Administrador Unico,Administrative,HR


So we can see that the department Label "Other" dominates. This is the main couse of the low accuracy. Maybe we should get the Label Other correct.

As an optional extension, we take the information from previous positions to provide additional context for active roles labeled as "Other", because its unlikely to change the Department.

Lets only look at the historical jobs which have no Label "Other" and which are INACTIVE 

In [38]:
from collections import Counter

def dept_from_history(cv):
    past_departments = [
        job["department"]
        for job in cv
        if job["status"] == "INACTIVE" and job["department"] != "Other"
    ]
    if not past_departments:
        return None
    return Counter(past_departments).most_common(1)[0][0]

In [39]:
person_to_inferred = {}

for person_id, cv in enumerate(cvs):
    inferred = dept_from_history(cv)
    if inferred is not None:
        person_to_inferred[person_id] = inferred

In [40]:
df_active["department_fixed"] = df_active["department"]

mask_other = df_active["department_fixed"] == "Other"

df_active.loc[mask_other, "department_fixed"] = (
    df_active.loc[mask_other, "person_id"]
    .map(person_to_inferred)
    .fillna("Other")
)

In [41]:
changed_rate = (df_active["department"] != df_active["department_fixed"]).mean()

coverage_other = (
    df_active.loc[df_active["department"] == "Other", "department_fixed"]
    .ne("Other")
    .mean()
)

changed_rate, coverage_other

(np.float64(0.27287319422150885), np.float64(0.4941860465116279))

In [42]:
embedding_accuracy_history = (
    df_active["department_fixed"] == df_active["department_pred_embedding"]
).mean()

embedding_accuracy_history

np.float64(0.11556982343499198)

In [43]:
HIGH_LEVEL_MAP = {
    # Tech
    "IT": "Tech",
    "Information Technology": "Tech",

    # Commercial
    "Marketing": "Commercial",
    "Sales": "Commercial",
    "Business Development": "Commercial",
    "Project Management": "Commercial",
    "Consulting": "Commercial",

    # Corporate
    "HR": "Corporate",
    "Human Resources": "Corporate",
    "Finance": "Corporate",
    "Administrative": "Corporate",

    # Other
    "Other": "Other"
}

In [44]:
df_active["department_fixed_cluster"] = df_active["department_fixed"].map(HIGH_LEVEL_MAP)
df_active["department_pred_cluster"] = df_active["department_pred_embedding"].map(HIGH_LEVEL_MAP)

In [45]:
hierarchical_accuracy = (
    df_active["department_fixed_cluster"] ==
    df_active["department_pred_cluster"]
).mean()

hierarchical_accuracy

np.float64(0.36436597110754415)

In [46]:
df_active[[
    "person_id",
    "position",
    "department",                  # ursprüngliche Wahrheit
    "department_fixed",            # nach History-Fix
    "department_pred_embedding"    # Modell
]].head(40)

,person_id,position,department,department_fixed,department_pred_embedding
0,0,Prokurist,Other,Other,HR
1,0,CFO,Other,Other,Finance
2,0,Betriebswirtin,Other,Other,IT
3,0,Prokuristin,Other,Other,HR
4,0,CFO,Other,Other,Finance
5,1,Solutions Architect,Information Technology,Information Technology,Project Management
6,2,Medizintechnik Beratung,Consulting,Consulting,Marketing
7,3,Director expansión de negocio.,Business Development,Business Development,Project Management
8,3,Gerente comercial,Sales,Sales,Marketing
9,3,Administrador Unico,Administrative,Administrative,HR
